# Ablation 4d: Minimal Smoke Test

**Purpose**: Check the curriculum learning assumptions in 2-3 minutes.

**Tests**:
1. Package installation (PyTorch 2.4, ultralytics)
2. Model loading with PyTorch 2.4
3. Inference on DDI image (hard negative)
4. Inference on wound image (positive)
5. Dice calculation

**Runtime**: 2-3 minutes (CPU only)
**Success**: all 5 tests pass. Next: full smoke test.
**Failure**: any test fails. Record the exact error.

## Test 1: Package Installation

PyTorch 2.4, ultralytics 8.0.196 and NumPy <2.0 install correctly.

In [ ]:
# Install packages with FIXED ultralytics version (8.3.x has bug fix)
# Bug: ultralytics 8.0.196 has AttributeError: 'Segment' object has no attribute 'detect'
# Fix: Upgrade to 8.3.40+ (bug fixed in 8.3.x series)
!pip install -q torch==2.4.1 torchvision==0.19.1 'ultralytics>=8.3.40' 'numpy<2.0'

# Validate installation
import importlib.util
assert importlib.util.find_spec('ultralytics') is not None, "ultralytics not installed"
assert importlib.util.find_spec('torch') is not None, "torch not installed"

import torch
import ultralytics
print(f'PyTorch: {torch.__version__}')
print(f'Ultralytics: {ultralytics.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
print('\nOK TEST 1 PASSED: All packages installed successfully')

## Test 2: Model Loading

Validate baseline model loads correctly with PyTorch 2.4.

In [ ]:
from ultralytics import YOLO
from pathlib import Path

# Load baseline model
model_path = '/kaggle/input/baseline-outputs/yolo_training/seed_42/weights/best.pt'
assert Path(model_path).exists(), f"Model not found: {model_path}"

model = YOLO(model_path)
print(f'Model loaded: {type(model.model)}')
print(f'Model task: {model.task}')
print('\nOK TEST 2 PASSED: Model loads successfully with PyTorch 2.4')

## Test 3: DDI Inference

Validate inference on hard negative (DDI) image.

**Check**: whether the segmentation model has a .boxes.conf attribute.

In [ ]:
import os

# Get first DDI image
ddi_dir = '/kaggle/input/ddi-diverse-dermatology'
assert Path(ddi_dir).exists(), f"DDI dataset not found: {ddi_dir}"

ddi_images = [f for f in os.listdir(ddi_dir) if f.endswith('.png') or f.endswith('.jpg')]
assert len(ddi_images) > 0, "No DDI images found"

test_img = Path(ddi_dir) / ddi_images[0]
print(f'Testing on: {test_img.name}')

# Run inference
results = model(str(test_img), verbose=False)

# Inspect result structure
print(f'\nResult structure:')
print(f'  Type: {type(results[0])}')
print(f'  Has boxes: {results[0].boxes is not None}')

if results[0].boxes is not None:
    print(f'  Boxes type: {type(results[0].boxes)}')
    print(f'  Has conf attr: {hasattr(results[0].boxes, "conf")}')
    
    if hasattr(results[0].boxes, 'conf'):
        conf = results[0].boxes.conf
        print(f'  Confidence shape: {conf.shape}')
        print(f'  Max confidence: {float(conf.max().cpu().numpy()):.4f}')
        print('\nOK TEST 3 PASSED: DDI inference works, .boxes.conf exists')
    else:
        print('\nFAIL TEST 3 FAILED: .boxes.conf does not exist')
        print('  Need alternative method to extract confidence')
else:
    print('  No boxes detected (expected for background image)')
    print('\nOK TEST 3 PASSED: DDI inference works (no detections on background)')

## Test 4: Wound Inference

Validate inference on wound image.

**Check**: whether segmentation masks can be extracted.

In [ ]:
# Get first wound image
wound_dir = '/kaggle/input/baseline-outputs/data_wound_seg/test_images'
assert Path(wound_dir).exists(), f"Wound images not found: {wound_dir}"

wound_images = [f for f in os.listdir(wound_dir) if f.endswith('.png') or f.endswith('.jpg')]
assert len(wound_images) > 0, "No wound images found"

test_img = Path(wound_dir) / wound_images[0]
print(f'Testing on: {test_img.name}')

# Run inference
results = model(str(test_img), verbose=False)

# Inspect masks
print(f'\nMask structure:')
print(f'  Has masks: {results[0].masks is not None}')

if results[0].masks is not None:
    print(f'  Masks type: {type(results[0].masks)}')
    print(f'  Number of masks: {len(results[0].masks)}')
    print(f'  Mask data shape: {results[0].masks.data[0].shape}')
    print('\nOK TEST 4 PASSED: Wound inference works, masks extractable')
else:
    print('\nFAIL TEST 4 FAILED: No masks detected on wound image')
    print('  This is unexpected - baseline should detect wounds')

## Test 5: Dice Calculation

Validate Dice calculation pipeline (mask loading, resizing, scoring).

In [ ]:
from PIL import Image
import numpy as np
import cv2

# Load ground truth mask
mask_name = wound_images[0]
mask_path = Path('/kaggle/input/baseline-outputs/data_wound_seg/test_masks') / mask_name
assert mask_path.exists(), f"GT mask not found: {mask_path}"

gt_mask = np.array(Image.open(mask_path).convert('L'))
gt_mask = (gt_mask > 127).astype(np.uint8)
print(f'GT mask shape: {gt_mask.shape}')
print(f'GT mask pixels: {gt_mask.sum()} positive')

# Get predicted mask
if results[0].masks is not None and len(results[0].masks) > 0:
    pred_mask_data = results[0].masks.data[0].cpu().numpy()
    pred_mask = cv2.resize(pred_mask_data, (gt_mask.shape[1], gt_mask.shape[0]))
    pred_mask = (pred_mask > 0.5).astype(np.uint8)
    print(f'Pred mask shape: {pred_mask.shape}')
    print(f'Pred mask pixels: {pred_mask.sum()} positive')
    
    # Calculate Dice
    intersection = np.logical_and(pred_mask, gt_mask).sum()
    union = pred_mask.sum() + gt_mask.sum()
    dice = 2.0 * intersection / union if union > 0 else 1.0
    
    print(f'\nDice score: {dice:.4f}')
    print('OK TEST 5 PASSED: Dice calculation works')
else:
    print('\nFAIL TEST 5 FAILED: Cannot calculate Dice (no predicted mask)')

## Test Summary

In [ ]:
import json
from datetime import datetime

test_summary = {
    'notebook': 'ablation_4d_minimal_smoke',
    'timestamp': datetime.now().isoformat(),
    'tests': [
        {'test': 1, 'name': 'Package Installation', 'status': 'Check output above'},
        {'test': 2, 'name': 'Model Loading', 'status': 'Check output above'},
        {'test': 3, 'name': 'DDI Inference', 'status': 'Check output above'},
        {'test': 4, 'name': 'Wound Inference', 'status': 'Check output above'},
        {'test': 5, 'name': 'Dice Calculation', 'status': 'Check output above'}
    ],
    'runtime_target': '2-3 minutes',
    'next_step': 'If all tests pass → deploy full smoke test (9 epochs, 15-20 min)',
    'on_failure': 'Report exact error, update assumptions, fix root cause'
}

with open('/kaggle/working/test_summary.json', 'w') as f:
    json.dump(test_summary, f, indent=2)

print('\n' + '='*60)
print('MINIMAL SMOKE TEST COMPLETE')
print('='*60)
print('\nReview test results above:')
print('  - All tests should show OK PASSED')
print('  - Any FAIL FAILED indicates assumption violation')
print('\nNext steps:')
print('  - If ALL pass: Deploy full smoke test')
print('  - If ANY fail: Debug root cause before proceeding')
print('\nOutputs:')
print('  - test_summary.json')